# Cohezion ARC-AGI-3 Offline Trajectory Miner
Simulates offline levels with Zobrist cycle evasion and archives solved trajectories.


In [ ]:
# 1. Install arc-agi runtime from offline competition wheelhouse
import os, sys, glob, subprocess

print("Installing arc-agi runtime from offline wheelhouse...")
wheel_dirs = glob.glob("/kaggle/input/**/arc_agi_3_wheels", recursive=True)
if wheel_dirs:
    wheel_dir = wheel_dirs[0]
    print(f"Found wheels directory: {wheel_dir}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", wheel_dir, "arc-agi", "python-dotenv", "pillow"], check=False)
else:
    print("arc_agi_3_wheels not found in /kaggle/input; dependencies should be pre-installed.")


In [ ]:
# 2. Cohezion ARC-AGI-3 SOTA v27: Monadic Markov Cycle Evasion + Zobrist State Hashing
# Lineage: v15 (0.26) -> v26 (0.13) -> v27 Monadic Markov Anti-Cycle Engine
import hashlib
import json
import logging
import os
import random
import sys
import time
from collections import deque
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Set
import numpy as np

import arc_agi
import arcengine
from arcengine import GameAction, GameState, FrameDataRaw

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("CohezionARC3")

# Precomputed 64-bit SplitMix64 Zobrist table for 64x64 grids across 10 discrete colors
RNG = np.random.default_rng(20261002)
ZOBRIST_TABLE = RNG.integers(0, (1 << 62) - 1, size=(64, 64, 10), dtype=np.int64)

OPPOSITE_ACTION = {1: 2, 2: 1, 3: 4, 4: 3}  # Up <-> Down, Left <-> Right

def hash_frame(frame_layers: list) -> int:
    """Vectorized 64-bit Zobrist XOR hash for O(1) grid state comparison."""
    if not frame_layers or frame_layers[0] is None:
        return 0
    grid = frame_layers[0]
    h, w = grid.shape
    if h > 64 or w > 64:
        return int(hashlib.md5(np.ascontiguousarray(grid).tobytes()).hexdigest()[:15], 16)
    colors = np.clip(grid, 0, 9)
    # Fast bitwise XOR reduction across the active grid slice
    r_idx = np.arange(h)[:, None]
    c_idx = np.arange(w)
    return int(np.bitwise_xor.reduce(ZOBRIST_TABLE[:h, :w, :][r_idx, c_idx, colors].ravel()))

def find_components(grid: np.ndarray) -> list[dict[str, Any]]:
    """Extract foreground connected components sorted by color rarity ascending, then size ascending."""
    h, w = grid.shape
    visited = np.zeros((h, w), dtype=bool)
    components = []
    vals, counts = np.unique(grid, return_counts=True)
    bg_color = vals[np.argmax(counts)]
    color_freq = dict(zip(vals, counts))

    for r in range(h):
        for c in range(w):
            if visited[r, c] or grid[r, c] == bg_color:
                continue
            color = int(grid[r, c])
            q = deque([(r, c)])
            visited[r, c] = True
            pixels = []
            while q:
                cr, cc = q.popleft()
                pixels.append((cr, cc))
                for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nr, nc = cr + dr, cc + dc
                    if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc] and grid[nr, nc] == color:
                        visited[nr, nc] = True
                        q.append((nr, nc))
            rs = [p[0] for p in pixels]
            cs = [p[1] for p in pixels]
            components.append({
                "color": color,
                "color_count": int(color_freq[color]),
                "size": len(pixels),
                "center_r": int(sum(rs) / len(rs)),
                "center_c": int(sum(cs) / len(cs)),
                "bbox": (min(rs), min(cs), max(rs), max(cs)),
            })

    # Strict Action Economy: Target rarest single-pixel and small-sprite affordances (key, player, door) first!
    components.sort(key=lambda c: (c["color_count"], c["size"]))
    return components

class CohezionDirectedRarityAgent:
    """Affordance-targeted search agent with Markov cycle evasion optimizing quadratic action economy."""
    def __init__(self, game_id: str, seed: int = 42):
        self.game_id = game_id
        self.rng = random.Random(seed + hash(game_id) % 100000)
        self.transition_model: dict[int, dict[int, int]] = {}
        self.state_visits: dict[int, int] = {}
        self.state_action_visits: dict[tuple[int, int], int] = {}
        self.clicked_targets: dict[int, set[tuple[int, int]]] = {}
        self.last_state_hash: Optional[int] = None
        self.last_action_id: Optional[int] = None
        self.last_frame: Optional[np.ndarray] = None
        self.player_pos: Optional[tuple[int, int]] = None
        self.step_count = 0
        self.consecutive_loops = 0
        self.state_history: deque[int] = deque(maxlen=24)
        self.recent_hashes: deque[int] = deque(maxlen=8)

    def choose_action(self, obs: FrameDataRaw) -> tuple[GameAction, Optional[dict[str, Any]]]:
        self.step_count += 1
        curr_hash = hash_frame(obs.frame)
        self.state_visits[curr_hash] = self.state_visits.get(curr_hash, 0) + 1
        grid = obs.frame[0] if obs.frame else None

        # Track avatar position from frame delta on directional movement
        if self.last_frame is not None and grid is not None and self.last_action_id in (1, 2, 3, 4):
            diff = (grid != self.last_frame)
            if 0 < np.sum(diff) <= 64:
                coords = np.argwhere(diff)
                self.player_pos = (int(np.mean(coords[:, 0])), int(np.mean(coords[:, 1])))

        self.last_frame = grid.copy() if grid is not None else None

        # Track state transitions and consecutive self-loops (wall collisions & orbit traps)
        is_cycle = False
        if self.last_state_hash is not None and self.last_action_id is not None:
            if self.last_state_hash not in self.transition_model:
                self.transition_model[self.last_state_hash] = {}
            self.transition_model[self.last_state_hash][self.last_action_id] = curr_hash
            sa_pair = (self.last_state_hash, self.last_action_id)
            self.state_action_visits[sa_pair] = self.state_action_visits.get(sa_pair, 0) + 1

            # 1-cycle check (wall bump)
            if curr_hash == self.last_state_hash:
                self.consecutive_loops += 1
                is_cycle = True
            # 2-cycle check (ping-pong oscillation s_A <-> s_B)
            elif len(self.state_history) >= 2 and curr_hash == self.state_history[-2]:
                self.consecutive_loops += 1
                is_cycle = True
            # 3-to-6 cycle check (orbit trap)
            elif curr_hash in self.recent_hashes:
                self.consecutive_loops += 1
                is_cycle = True
            else:
                self.consecutive_loops = 0

        self.state_history.append(curr_hash)
        self.recent_hashes.append(curr_hash)

        avail = obs.available_actions
        simple_ids = [a for a in avail if a not in (0, 6, 7)]
        has_click = 6 in avail

        if curr_hash not in self.clicked_targets:
            self.clicked_targets[curr_hash] = set()

        comps = find_components(grid) if grid is not None else []

        # 1. Click handling: Trigger on unclicked affordances, or breakout if trapped in loops
        if has_click and (not simple_ids or self.consecutive_loops >= 2 or self.rng.random() < 0.35):
            unclicked = [c for c in comps if (c["center_c"], c["center_r"]) not in self.clicked_targets[curr_hash]]
            if unclicked:
                target = unclicked[0]
                coord = (target["center_c"], target["center_r"])
                self.clicked_targets[curr_hash].add(coord)
                self.last_state_hash = curr_hash
                self.last_action_id = 6
                return GameAction.ACTION6, {"x": coord[0], "y": coord[1]}
            elif comps and not simple_ids:
                # Cycle click positions within bounding boxes to prevent lock
                target = self.rng.choice(comps)
                r_min, c_min, r_max, c_max = target["bbox"]
                coord = (self.rng.randint(c_min, c_max), self.rng.randint(r_min, r_max))
                self.last_state_hash = curr_hash
                self.last_action_id = 6
                return GameAction.ACTION6, {"x": coord[0], "y": coord[1]}

        # 2. Directed Movement handling with Markov Cycle Evasion
        if simple_ids:
            known = self.transition_model.get(curr_hash, {})

            # Compute preference ranking towards rarest affordance
            preferred_scores = {a: 0.0 for a in simple_ids}
            if self.player_pos is not None and comps:
                pr, pc = self.player_pos
                target = comps[0]
                tr, tc = target["center_r"], target["center_c"]
                dr, dc = tr - pr, tc - pc
                if abs(dr) > abs(dc):
                    preferred = [1 if dr < 0 else 2, 3 if dc < 0 else 4]
                else:
                    preferred = [3 if dc < 0 else 4, 1 if dr < 0 else 2]
                for idx, act_id in enumerate(preferred):
                    if act_id in preferred_scores:
                        preferred_scores[act_id] -= (10.0 - idx * 4.0)

            # Score each candidate action with cycle and repetition penalties
            action_penalties = {}
            for act_id in simple_ids:
                penalty = preferred_scores[act_id]

                # Immediate reverse penalty if last action formed a 2-cycle
                if is_cycle and self.last_action_id in OPPOSITE_ACTION:
                    if act_id == OPPOSITE_ACTION[self.last_action_id]:
                        penalty += 50.0

                if act_id in known:
                    next_s = known[act_id]
                    # Self loop / wall bump penalty
                    if next_s == curr_hash:
                        penalty += 100.0
                    # Recent orbit trap penalty
                    if next_s in self.recent_hashes:
                        penalty += 35.0 * (self.recent_hashes.count(next_s) + 1)
                    # General state visit penalty
                    penalty += 2.0 * self.state_visits.get(next_s, 0)
                else:
                    # Untried actions get an exploration bonus
                    penalty -= 5.0

                sa_pair = (curr_hash, act_id)
                penalty += 1.0 * self.state_action_visits.get(sa_pair, 0)
                action_penalties[act_id] = penalty

            # Pick action minimizing total penalty
            chosen_id = min(simple_ids, key=lambda a: action_penalties[a])

            self.last_state_hash = curr_hash
            self.last_action_id = chosen_id
            return GameAction.from_id(chosen_id), None

        fallback_id = avail[0] if avail else 0
        return GameAction.from_id(fallback_id), None

# Backward compatibility aliases
CohezionRaritySearchAgent = CohezionDirectedRarityAgent
CohezionGoExploreAgent = CohezionDirectedRarityAgent
CohezionMonadicMarkovRarityAgent = CohezionDirectedRarityAgent


In [ ]:
# 3. Offline Environment Trajectory Mining Loop
import os, sys, glob, time, json
from pathlib import Path
import pandas as pd
from arcengine import GameAction, GameState

print("=== ARC-AGI-3 Offline Trajectory Mining Runner Initialized ===")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

env_dirs = glob.glob("/kaggle/input/**/environment_files", recursive=True)
env_dir = env_dirs[0] if env_dirs else "data/arc_prize/environment_files"
print(f"Searching environments in: {env_dir}")

arcade = arc_agi.Arcade(
    operation_mode=arc_agi.OperationMode.OFFLINE,
    environments_dir=env_dir,
)
card_id = arcade.create_scorecard()
print(f"Created offline scorecard: {card_id}")

available_games = ["ar25", "bp35", "cd82", "cn04", "dc22", "ft09", "g50t", "ka59", "lf52", "lp85", "ls20", "m0r0", "r11l", "re86", "s5i5", "sb26", "sc25", "sk48", "sp80", "su15", "tn36", "tr87", "tu93", "vc33", "wa30"]
print(f"Targeting {len(available_games)} discrete arcade game environments.")

records = []
solved_count = 0
total_steps_executed = 0
t0 = time.time()

for i, game_name in enumerate(available_games):
    elapsed = time.time() - t0
    if elapsed > 10 * 3600:
        print("Reached compute safety cap. Finalizing parquet.")
        break
    try:
        env = arcade.make(game_name, scorecard_id=card_id)
        if env is None:
            continue
        obs = env.reset()
        agent = CohezionDirectedRarityAgent(game_id=game_name)
        step_history = []
        step = 0
        max_steps = 80
        won = False
        while obs.state == GameState.PLAYING and step < max_steps:
            act, data = agent.choose_action(obs)
            obs = env.step(act, data=data)
            step_history.append({
                "step": step,
                "action": int(act.value) if hasattr(act, "value") else int(act),
                "data": data,
                "state": str(obs.state)
            })
            step += 1
            total_steps_executed += 1
            if obs.state == GameState.WON:
                won = True
                solved_count += 1
                break
        records.append({
            "game_name": game_name,
            "steps": step,
            "won": won,
            "final_state": str(obs.state),
            "trajectory": json.dumps(step_history)
        })
        print(f"[{i+1}/{len(available_games)}] Game: {game_name} | State: {obs.state} | Steps: {step} | Total Solved: {solved_count}")
    except Exception as e:
        print(f"Exception on game {game_name}: {e}")

if records:
    df_traj = pd.DataFrame(records)
    pq_path = OUT_DIR / "arc3_solved_trajectories.parquet"
    df_traj.to_parquet(pq_path, index=False)
    print(f"✓ Exported {len(df_traj)} game trajectories ({solved_count} solved) -> {pq_path}")
else:
    df_dummy = pd.DataFrame([{"game_name": "placeholder", "steps": 0, "won": False, "final_state": "PLAYING", "trajectory": "[]"}])
    df_dummy.to_parquet(OUT_DIR / "arc3_solved_trajectories.parquet", index=False)
    print("Exported placeholder parquet.")
